In [3]:
import pandas as pd

# Load lại data từ file pickle
df = pd.read_pickle(r"D:\data_driven_marketing\df_full.pkl")

# Kiểm tra nhanh
print(df.shape)
df.head()

(1608337, 59)


,channelGrouping,date,fullVisitorId,visitId,visitNumber,visitStartTime,device_browser,device_operatingSystem,device_isMobile,device_deviceCategory,...,Date_Is_month_start,Date_Is_quarter_end,Date_Is_quarter_start,Date_Is_year_end,Date_Is_year_start,visitStartTime_datetime,Date_Hour,transactionRevenue,transactionRevenue_dollar,target_log_revenue
0,Organic Search,2017-10-16,3162355547410993243,1508198450,1,1508198450,Firefox,Windows,False,desktop,...,0,0,0,0,0,2017-10-17 00:00:50,0,0.0,0.0,0.0
1,Referral,2017-10-16,8934116514970143966,1508176307,6,1508176307,Chrome,Chrome OS,False,desktop,...,0,0,0,0,0,2017-10-16 17:51:47,17,0.0,0.0,0.0
2,Direct,2017-10-16,7992466427990357681,1508201613,1,1508201613,Chrome,Android,True,mobile,...,0,0,0,0,0,2017-10-17 00:53:33,0,0.0,0.0,0.0
3,Organic Search,2017-10-16,9075655783635761930,1508169851,1,1508169851,Chrome,Windows,False,desktop,...,0,0,0,0,0,2017-10-16 16:04:11,16,0.0,0.0,0.0
4,Organic Search,2017-10-16,6960673291025684308,1508190552,1,1508190552,Chrome,Windows,False,desktop,...,0,0,0,0,0,2017-10-16 21:49:12,21,0.0,0.0,0.0


In [5]:
df.columns

Index(['channelGrouping', 'date', 'fullVisitorId', 'visitId', 'visitNumber',
       'visitStartTime', 'device_browser', 'device_operatingSystem',
       'device_isMobile', 'device_deviceCategory', 'geoNetwork_continent',
       'geoNetwork_subContinent', 'geoNetwork_country', 'geoNetwork_region',
       'geoNetwork_metro', 'geoNetwork_city', 'geoNetwork_networkDomain',
       'totals_visits', 'totals_hits', 'totals_pageviews', 'totals_bounces',
       'totals_newVisits', 'totals_sessionQualityDim', 'totals_timeOnSite',
       'totals_transactions', 'totals_transactionRevenue',
       'totals_totalTransactionRevenue', 'trafficSource_campaign',
       'trafficSource_source', 'trafficSource_medium', 'trafficSource_keyword',
       'trafficSource_referralPath', 'trafficSource_isTrueDirect',
       'trafficSource_adContent', 'trafficSource_adwordsClickInfo.page',
       'trafficSource_adwordsClickInfo.slot',
       'trafficSource_adwordsClickInfo.gclId',
       'trafficSource_adwordsClickIn

In [6]:
FEATURE_DAYS = 168
TARGET_DAYS = 92
STEP_DAYS = 60

In [9]:
import pandas as pd
import numpy as np

df = df.copy()

# =========================
# 1. Chuẩn hóa cột thời gian
# =========================
if "visitStartTime_datetime" in df.columns:
    df["event_date"] = pd.to_datetime(df["visitStartTime_datetime"])
elif "date" in df.columns:
    df["event_date"] = pd.to_datetime(df["date"])
else:
    df["event_date"] = pd.to_datetime(df["visitStartTime"], unit="s")

# =========================
# 2. Chọn cột revenue
# =========================
# Ưu tiên transactionRevenue vì data bạn có cột này
if "transactionRevenue" in df.columns:
    revenue_col = "transactionRevenue"
elif "totals_transactionRevenue" in df.columns:
    revenue_col = "totals_transactionRevenue"
else:
    raise ValueError("Không tìm thấy cột revenue phù hợp.")

df[revenue_col] = pd.to_numeric(df[revenue_col], errors="coerce").fillna(0)

# =========================
# 3. Ép numeric cho các cột quan trọng
# =========================
numeric_cols = [
    "visitNumber",
    "totals_visits",
    "totals_hits",
    "totals_pageviews",
    "totals_bounces",
    "totals_newVisits",
    "totals_sessionQualityDim",
    "totals_timeOnSite",
    "totals_transactions",
    "totals_transactionRevenue",
    "totals_totalTransactionRevenue",
    "transactionRevenue",
    "transactionRevenue_dollar",
    "customDimensions_count",
    "customDimensions_index",
    "Date_Year",
    "Date_Month",
    "Date_Day",
    "Date_Dayofweek",
    "Date_Dayofyear",
    "Date_Week",
    "Date_Hour"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print("Data shape:", df.shape)
print("Date range:", df["event_date"].min(), "→", df["event_date"].max())
print("Revenue column:", revenue_col)

Data shape: (1608337, 60)
Date range: 2016-08-02 07:00:09 → 2018-05-01 06:56:58
Revenue column: transactionRevenue


In [10]:
def create_no_gap_time_fold_by_step(
    data,
    k,
    feature_days=168,
    target_days=92,
    step_days=60,
    revenue_col="transactionRevenue",
    base_date=None
):
    data = data.copy()

    if base_date is None:
        base_date = data["event_date"].min()
    else:
        base_date = pd.to_datetime(base_date)

    # =========================
    # 1. Xác định window
    # =========================
    feature_start = base_date + pd.Timedelta(days=step_days * (k - 1))
    feature_end = feature_start + pd.Timedelta(days=feature_days)

    target_start = feature_end
    target_end = target_start + pd.Timedelta(days=target_days)

    if target_end > data["event_date"].max():
        print(
            f"Skip Fold {k}: target_end {target_end.date()} "
            f"> max date {data['event_date'].max().date()}"
        )
        return None

    df_feat = data[
        (data["event_date"] >= feature_start) &
        (data["event_date"] < feature_end)
    ].copy()

    df_target = data[
        (data["event_date"] >= target_start) &
        (data["event_date"] < target_end)
    ].copy()

    print(
        f"Fold {k}: "
        f"Feature {feature_start.date()} → {(feature_end - pd.Timedelta(days=1)).date()} | "
        f"Target {target_start.date()} → {(target_end - pd.Timedelta(days=1)).date()} | "
        f"Feature rows: {len(df_feat)} | Target rows: {len(df_target)}"
    )

    # =========================
    # 2. Tạo target
    # target_log_revenue = log(1 + sum revenue trong target window)
    # ret = 1 nếu user có revenue > 0 trong target window
    # =========================
    target = (
        df_target
        .groupby("fullVisitorId")[revenue_col]
        .sum()
        .reset_index()
        .rename(columns={revenue_col: "target_revenue"})
    )

    target["target_log_revenue"] = np.log1p(target["target_revenue"])
    target["ret"] = (target["target_revenue"] > 0).astype(int)

    # =========================
    # 3. Numerical aggregation từ Feature Window
    # =========================
    agg_dict = {}

    if "visitId" in df_feat.columns:
        agg_dict["visitId"] = "nunique"

    feature_numeric_cols = [
        "visitNumber",
        "totals_visits",
        "totals_hits",
        "totals_pageviews",
        "totals_bounces",
        "totals_newVisits",
        "totals_sessionQualityDim",
        "totals_timeOnSite",
        "totals_transactions",
        revenue_col
    ]

    for col in feature_numeric_cols:
        if col in df_feat.columns:
            agg_dict[col] = ["sum", "mean", "min", "max", "median", "std"]

    X_num = df_feat.groupby("fullVisitorId").agg(agg_dict)

    X_num.columns = [
        "_".join(col).strip() if isinstance(col, tuple) else col
        for col in X_num.columns
    ]

    X_num = X_num.reset_index()

    if "visitId_nunique" in X_num.columns:
        X_num = X_num.rename(columns={"visitId_nunique": "session_cnt"})

    X_num = X_num.fillna(0)

    # =========================
    # 4. Time features
    # =========================
    user_time = (
        df_feat
        .groupby("fullVisitorId")["event_date"]
        .agg(["min", "max", "nunique"])
        .reset_index()
        .rename(columns={
            "min": "first_visit_date",
            "max": "last_visit_date",
            "nunique": "unique_date_num"
        })
    )

    user_time["first_ses_from_period_start"] = (
        user_time["first_visit_date"] - feature_start
    ).dt.days

    user_time["last_ses_from_period_end"] = (
        feature_end - user_time["last_visit_date"]
    ).dt.days

    user_time["interval_dates"] = (
        user_time["last_visit_date"] - user_time["first_visit_date"]
    ).dt.days

    X = X_num.merge(user_time, on="fullVisitorId", how="left")

    # =========================
    # 5. Categorical features
    # lấy giá trị ở lần visit cuối cùng trong Feature Window
    # =========================
    categorical_cols = [
        "channelGrouping",
        "device_browser",
        "device_operatingSystem",
        "device_isMobile",
        "device_deviceCategory",
        "geoNetwork_continent",
        "geoNetwork_subContinent",
        "geoNetwork_country",
        "geoNetwork_region",
        "geoNetwork_metro",
        "geoNetwork_city",
        "geoNetwork_networkDomain",
        "trafficSource_campaign",
        "trafficSource_source",
        "trafficSource_medium",
        "trafficSource_keyword",
        "trafficSource_referralPath",
        "trafficSource_isTrueDirect",
        "trafficSource_adContent",
        "trafficSource_adwordsClickInfo.page",
        "trafficSource_adwordsClickInfo.slot",
        "trafficSource_adwordsClickInfo.gclId",
        "trafficSource_adwordsClickInfo.adNetworkType",
        "trafficSource_adwordsClickInfo.isVideoAd",
        "customDimensions_value"
    ]

    categorical_cols = [c for c in categorical_cols if c in df_feat.columns]

    if len(categorical_cols) > 0:
        df_last = (
            df_feat
            .sort_values(["fullVisitorId", "event_date"])
            .groupby("fullVisitorId")
            .tail(1)
        )

        X_cat = df_last[["fullVisitorId"] + categorical_cols].copy()
        X = X.merge(X_cat, on="fullVisitorId", how="left")

    # =========================
    # 6. Merge X + y
    # User có trong feature window nhưng không có revenue target -> 0
    # =========================
    fold_df = X.merge(target, on="fullVisitorId", how="left")

    fold_df["target_revenue"] = fold_df["target_revenue"].fillna(0)
    fold_df["target_log_revenue"] = fold_df["target_log_revenue"].fillna(0)
    fold_df["ret"] = fold_df["ret"].fillna(0).astype(int)

    fold_df["fold_id"] = k
    fold_df["feature_start"] = feature_start
    fold_df["feature_end"] = feature_end
    fold_df["target_start"] = target_start
    fold_df["target_end"] = target_end

    return fold_df

In [11]:
FEATURE_DAYS = 168
TARGET_DAYS = 92
STEP_DAYS = 60

base_date = df["event_date"].min()

fold_dfs = []

k = 1
while True:
    fold_k = create_no_gap_time_fold_by_step(
        data=df,
        k=k,
        feature_days=FEATURE_DAYS,
        target_days=TARGET_DAYS,
        step_days=STEP_DAYS,
        revenue_col=revenue_col,
        base_date=base_date
    )

    if fold_k is None:
        break

    fold_dfs.append(fold_k)
    k += 1

data_folds = pd.concat(fold_dfs, axis=0).reset_index(drop=True)

print("Number of folds:", data_folds["fold_id"].nunique())
print("All folds shape:", data_folds.shape)

Fold 1: Feature 2016-08-02 → 2017-01-16 | Target 2017-01-17 → 2017-04-18 | Feature rows: 438346 | Target rows: 194480
Fold 2: Feature 2016-10-01 → 2017-03-17 | Target 2017-03-18 → 2017-06-17 | Feature rows: 426446 | Target rows: 191878
Fold 3: Feature 2016-11-30 → 2017-05-16 | Target 2017-05-17 → 2017-08-16 | Feature rows: 359751 | Target rows: 197858
Fold 4: Feature 2017-01-29 → 2017-07-15 | Target 2017-07-16 → 2017-10-15 | Feature rows: 351811 | Target rows: 244583
Fold 5: Feature 2017-03-30 → 2017-09-13 | Target 2017-09-14 → 2017-12-14 | Feature rows: 372110 | Target rows: 278607
Fold 6: Feature 2017-05-29 → 2017-11-12 | Target 2017-11-13 → 2018-02-12 | Feature rows: 419531 | Target rows: 257610
Fold 7: Feature 2017-07-28 → 2018-01-11 | Target 2018-01-12 → 2018-04-13 | Feature rows: 468763 | Target rows: 255692
Skip Fold 8: target_end 2018-06-13 > max date 2018-05-01
Number of folds: 7
All folds shape: (2251631, 101)


In [12]:
fold_summary = (
    data_folds
    .groupby("fold_id")
    .agg(
        n_rows=("fullVisitorId", "count"),
        n_users=("fullVisitorId", "nunique"),
        n_buyers=("ret", "sum"),
        buyer_rate=("ret", "mean"),
        mean_target=("target_log_revenue", "mean"),
        max_target=("target_log_revenue", "max"),
        feature_start=("feature_start", "min"),
        feature_end=("feature_end", "max"),
        target_start=("target_start", "min"),
        target_end=("target_end", "max")
    )
    .reset_index()
)

fold_summary

,fold_id,n_rows,n_users,n_buyers,buyer_rate,mean_target,max_target,feature_start,feature_end,target_start,target_end
0,1,356690,356690,290,0.000813,0.014918,24.899596,2016-08-02 07:00:09,2017-01-17 07:00:09,2017-01-17 07:00:09,2017-04-19 07:00:09
1,2,347317,347317,361,0.001039,0.019028,24.740841,2016-10-01 07:00:09,2017-03-18 07:00:09,2017-03-18 07:00:09,2017-06-18 07:00:09
2,3,283651,283651,326,0.001149,0.020910,23.626948,2016-11-30 07:00:09,2017-05-17 07:00:09,2017-05-17 07:00:09,2017-08-17 07:00:09
3,4,277744,277744,403,0.001451,0.026110,24.436965,2017-01-29 07:00:09,2017-07-16 07:00:09,2017-07-16 07:00:09,2017-10-16 07:00:09
4,5,292611,292611,256,0.000875,0.015762,22.545973,2017-03-30 07:00:09,2017-09-14 07:00:09,2017-09-14 07:00:09,2017-12-15 07:00:09
5,6,328246,328246,273,0.000832,0.014918,21.405135,2017-05-29 07:00:09,2017-11-13 07:00:09,2017-11-13 07:00:09,2018-02-13 07:00:09
6,7,365372,365372,291,0.000796,0.014285,22.030977,2017-07-28 07:00:09,2018-01-12 07:00:09,2018-01-12 07:00:09,2018-04-14 07:00:09


In [13]:
max_fold = data_folds["fold_id"].max()

train_df = data_folds[data_folds["fold_id"] <= max_fold - 2].copy()
val_df = data_folds[data_folds["fold_id"] == max_fold - 1].copy()
test_df = data_folds[data_folds["fold_id"] == max_fold].copy()

print("Train folds:", sorted(train_df["fold_id"].unique()))
print("Validation fold:", sorted(val_df["fold_id"].unique()))
print("Test fold:", sorted(test_df["fold_id"].unique()))

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Validation fold: [np.int64(6)]
Test fold: [np.int64(7)]
Train shape: (1558013, 101)
Validation shape: (328246, 101)
Test shape: (365372, 101)


In [14]:
target_col = "target_log_revenue"

drop_cols = [
    "fullVisitorId",
    "target_revenue",
    "target_log_revenue",
    "ret",
    "fold_id",
    "feature_start",
    "feature_end",
    "target_start",
    "target_end",
    "first_visit_date",
    "last_visit_date"
]

X_train = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
y_train = train_df[target_col]

X_val = val_df.drop(columns=[c for c in drop_cols if c in val_df.columns])
y_val = val_df[target_col]

X_test = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
y_test = test_df[target_col]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (1558013, 90)
X_val: (328246, 90)
X_test: (365372, 90)


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

# =========================
# 1. Copy để tránh sửa nhầm data gốc
# =========================
X_train_clean = X_train.copy()
X_val_clean = X_val.copy()
X_test_clean = X_test.copy()

# =========================
# 2. Xác định numeric / categorical
# =========================
numeric_features = X_train_clean.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X_train_clean.select_dtypes(
    include=["object", "bool", "category", "string"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

# =========================
# 3. Clean categorical: fill NA + ép string
# =========================
for data_part in [X_train_clean, X_val_clean, X_test_clean]:
    for col in categorical_features:
        data_part[col] = (
            data_part[col]
            .replace({pd.NA: np.nan})
            .fillna("missing")
            .astype(str)
        )

# =========================
# 4. Pipeline numeric / categorical
# =========================
num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("ordinal", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, numeric_features),
        ("cat", cat_pipeline, categorical_features)
    ],
    remainder="drop"
)

# =========================
# 5. Random Forest baseline
# =========================
reg_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

reg_pipe = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", reg_model)
    ]
)

# =========================
# 6. Train
# =========================
reg_pipe.fit(X_train_clean, y_train)

val_pred = reg_pipe.predict(X_val_clean)

val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
val_r2 = r2_score(y_val, val_pred)

print("Validation RMSE:", val_rmse)
print("Validation R2:", val_r2)

Numeric features: 66
Categorical features: 24
Validation RMSE: 0.5111196772144164
Validation R2: 0.02774190387468989


In [18]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    r2_score
)

# =====================================================
# 1. Hàm đánh giá mô hình regression đầy đủ
# =====================================================

def evaluate_regression_model(model, X, y_true, dataset_name="Dataset"):
    """
    Đánh giá mô hình regression trên một tập dữ liệu.
    Phù hợp nếu target là log revenue, ví dụ target_log_revenue.
    """

    # Predict
    y_pred = model.predict(X)

    # Metrics cơ bản trên scale hiện tại của target
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    # Residual
    residuals = y_true - y_pred

    # Nếu target đang là log1p(revenue), có thể inverse về revenue gốc
    y_true_revenue = np.expm1(y_true)
    y_pred_revenue = np.expm1(y_pred)

    # Tránh giá trị âm sau inverse do mô hình dự đoán log âm
    y_pred_revenue = np.clip(y_pred_revenue, 0, None)

    revenue_rmse = np.sqrt(mean_squared_error(y_true_revenue, y_pred_revenue))
    revenue_mae = mean_absolute_error(y_true_revenue, y_pred_revenue)

    # Tổng hợp metrics
    metrics = {
        "dataset": dataset_name,
        "n_samples": len(y_true),

        "rmse_log": rmse,
        "mae_log": mae,
        "median_ae_log": medae,
        "r2_log": r2,

        "rmse_revenue_original_scale": revenue_rmse,
        "mae_revenue_original_scale": revenue_mae,

        "y_true_min_log": np.min(y_true),
        "y_true_mean_log": np.mean(y_true),
        "y_true_median_log": np.median(y_true),
        "y_true_max_log": np.max(y_true),

        "y_pred_min_log": np.min(y_pred),
        "y_pred_mean_log": np.mean(y_pred),
        "y_pred_median_log": np.median(y_pred),
        "y_pred_max_log": np.max(y_pred),

        "residual_mean": np.mean(residuals),
        "residual_std": np.std(residuals),
        "residual_min": np.min(residuals),
        "residual_25%": np.percentile(residuals, 25),
        "residual_50%": np.percentile(residuals, 50),
        "residual_75%": np.percentile(residuals, 75),
        "residual_max": np.max(residuals),
    }

    metrics_df = pd.DataFrame([metrics])

    # Bảng chi tiết từng quan sát
    detail_df = pd.DataFrame({
        "y_true_log": y_true,
        "y_pred_log": y_pred,
        "residual_log": residuals,
        "abs_error_log": np.abs(residuals),
        "squared_error_log": residuals ** 2,

        "y_true_revenue": y_true_revenue,
        "y_pred_revenue": y_pred_revenue,
        "residual_revenue": y_true_revenue - y_pred_revenue,
        "abs_error_revenue": np.abs(y_true_revenue - y_pred_revenue),
    })

    print("=" * 80)
    print(f"Evaluation result for: {dataset_name}")
    print("=" * 80)

    print("\nMain metrics:")
    display(metrics_df.T)

    print("\nPrediction detail sample:")
    display(detail_df.head(20))

    print("\nPrediction detail descriptive statistics:")
    display(detail_df.describe().T)

    return metrics_df, detail_df

In [19]:
# =====================================================
# 2. Evaluate train, validation, test
# =====================================================

train_metrics, train_detail = evaluate_regression_model(
    model=reg_pipe,
    X=X_train_clean,
    y_true=y_train,
    dataset_name="Train"
)

val_metrics, val_detail = evaluate_regression_model(
    model=reg_pipe,
    X=X_val_clean,
    y_true=y_val,
    dataset_name="Validation"
)

test_metrics, test_detail = evaluate_regression_model(
    model=reg_pipe,
    X=X_test_clean,
    y_true=y_test,
    dataset_name="Test"
)

Evaluation result for: Train

Main metrics:


,0
dataset,Train
n_samples,1558013
rmse_log,0.546608
mae_log,0.034131
median_ae_log,0.001498
r2_log,0.142241
rmse_revenue_original_scale,80152458.303749
mae_revenue_original_scale,353485.385103
y_true_min_log,0.0
y_true_mean_log,0.019079



Prediction detail sample:


,y_true_log,y_pred_log,residual_log,abs_error_log,squared_error_log,y_true_revenue,y_pred_revenue,residual_revenue,abs_error_revenue
0,0.0,0.001179,-0.001179,0.001179,0.000001,0.0,0.001180,-0.001180,0.001180
1,0.0,0.012189,-0.012189,0.012189,0.000149,0.0,0.012263,-0.012263,0.012263
2,0.0,0.001498,-0.001498,0.001498,0.000002,0.0,0.001499,-0.001499,0.001499
3,0.0,0.001498,-0.001498,0.001498,0.000002,0.0,0.001499,-0.001499,0.001499
4,0.0,0.001051,-0.001051,0.001051,0.000001,0.0,0.001052,-0.001052,0.001052
5,0.0,0.001051,-0.001051,0.001051,0.000001,0.0,0.001052,-0.001052,0.001052
6,0.0,0.016140,-0.016140,0.016140,0.000260,0.0,0.016271,-0.016271,0.016271
7,0.0,0.006052,-0.006052,0.006052,0.000037,0.0,0.006070,-0.006070,0.006070
8,0.0,0.001498,-0.001498,0.001498,0.000002,0.0,0.001499,-0.001499,0.001499
9,0.0,0.001051,-0.001051,0.001051,0.000001,0.0,0.001052,-0.001052,0.001052



Prediction detail descriptive statistics:


...

,0
dataset,Test
n_samples,365372
rmse_log,0.502191
mae_log,0.045463
median_ae_log,0.001743
r2_log,0.018992
rmse_revenue_original_scale,9257003.853703
mae_revenue_original_scale,108416.434039
y_true_min_log,0.0
y_true_mean_log,0.014285



Prediction detail sample:


,y_true_log,y_pred_log,residual_log,abs_error_log,squared_error_log,y_true_revenue,y_pred_revenue,residual_revenue,abs_error_revenue
1886259,0.0,0.025305,-0.025305,0.025305,0.000640,0.0,0.025628,-0.025628,0.025628
1886260,0.0,0.001747,-0.001747,0.001747,0.000003,0.0,0.001748,-0.001748,0.001748
1886261,0.0,0.005208,-0.005208,0.005208,0.000027,0.0,0.005222,-0.005222,0.005222
1886262,0.0,0.001051,-0.001051,0.001051,0.000001,0.0,0.001052,-0.001052,0.001052
1886263,0.0,0.022486,-0.022486,0.022486,0.000506,0.0,0.022741,-0.022741,0.022741
1886264,0.0,0.106614,-0.106614,0.106614,0.011367,0.0,0.112505,-0.112505,0.112505
1886265,0.0,0.005269,-0.005269,0.005269,0.000028,0.0,0.005283,-0.005283,0.005283
1886266,0.0,0.027561,-0.027561,0.027561,0.000760,0.0,0.027945,-0.027945,0.027945
1886267,0.0,0.001055,-0.001055,0.001055,0.000001,0.0,0.001056,-0.001056,0.001056
1886268,0.0,0.016149,-0.016149,0.016149,0.000261,0.0,0.016280,-0.016280,0.016280



Prediction detail descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
y_true_log,365372.0,0.014285,5.070294e-01,0.000000e+00,0.000000,0.000000,0.000000,2.203098e+01
y_pred_log,365372.0,0.032695,1.476475e-01,1.031212e-03,0.001057,0.001743,0.018380,1.542167e+01
residual_log,365372.0,-0.018409,5.018541e-01,-1.542167e+01,-0.018348,-0.001738,-0.001057,2.097797e+01
abs_error_log,365372.0,0.045463,5.001295e-01,1.031212e-03,0.001057,0.001743,0.018380,2.097797e+01
squared_error_log,365372.0,0.252196,8.407129e+00,1.063399e-06,0.000001,0.000003,0.000338,4.400752e+02
y_true_revenue,365372.0,108401.300592,9.256426e+06,0.000000e+00,0.000000,0.000000,0.000000,3.697700e+09
y_pred_revenue,365372.0,17.669626,8.368971e+03,1.031744e-03,0.001057,0.001744,0.018550,4.983610e+06
residual_revenue,365372.0,108383.630967,9.256382e+06,-4.983610e+06,-0.018518,-0.001740,-0.001057,3.697700e+09
abs_error_revenue,365372.0,108416.434039,9.256382e+06,1.031744e-03,0.001057,0.001744,0.018550,3.697700e+09
